# Concrete ML + FHE Cloud Resource Alert Detection

This notebook keeps the original plaintext XGBoost baseline and adds a Concrete ML
XGBoost model for fully homomorphic encrypted (FHE) inference.

Pipeline:

**CSV → preprocessing → plaintext baseline → Concrete ML → FHE compilation → encrypted inference → comparison**

In [ ]:
# CELL 1 — INSTALL DEPENDENCIES

!pip -q install concrete-ml xgboost scikit-learn pandas numpy joblib

In [ ]:
# CELL 2 — IMPORTS

import pandas as pd
import numpy as np
import joblib
import time
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier as XGBBaseline
from concrete.ml.sklearn import XGBClassifier as ConcreteXGBClassifier

print("Imports successful.")

In [ ]:
# CELL 3 — LOAD DATASET

DATA_PATH = "/content/FHE_ready_cloud_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# CELL 4 — BASIC CLEANING

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

TARGET = "resource_alert"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print("\nTarget distribution:")
print(df[TARGET].value_counts())

In [ ]:
# CELL 5 — TIMESTAMP / TIME FEATURES

if "timestamp" in df.columns:

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    )

    df["year"] = df["timestamp"].dt.year
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["day_of_year"] = df["timestamp"].dt.dayofyear

    df["week_of_year"] = (
        df["timestamp"]
        .dt.isocalendar()
        .week
        .astype(float)
    )

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    df.drop(columns=["timestamp"], inplace=True)

In [ ]:
# CELL 6 — REMOVE IDENTIFIER COLUMNS

id_columns = [
    "vm_id",
    "id",
    "ID",
    "record_id",
    "recordId"
]

for col in id_columns:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

print("Remaining columns:")
print(df.columns.tolist())

In [ ]:
# CELL 7 — PREPARE FEATURES AND TARGET

X = df.drop(columns=[TARGET])
y = df[TARGET].copy()

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

encoders = {}

for col in categorical_columns:
    X[col] = X[col].fillna("UNKNOWN").astype(str)

    encoder = LabelEncoder()
    X[col] = encoder.fit_transform(X[col])

    encoders[col] = encoder

In [ ]:
# CELL 8 — NUMERIC CLEANING

for col in X.columns:

    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

    X[col] = X[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

    X[col] = X[col].fillna(X[col].median())

X = X.astype(np.float32)

print("Feature matrix shape:", X.shape)
print("Feature dtypes:")
print(X.dtypes)

In [ ]:
# CELL 9 — TARGET ENCODING

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print("Target classes:")

for i, cls in enumerate(target_encoder.classes_):
    print(i, "=", cls)

num_classes = len(np.unique(y_encoded))

print("\nNumber of classes:", num_classes)

if num_classes != 2:
    raise ValueError(
        "This notebook expects a binary resource_alert target."
    )

In [ ]:
# CELL 10 — TRAIN / VALIDATION / TEST SPLIT

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples  :", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples   :", len(X_test))

## Part A — Plaintext XGBoost Baseline

This preserves the original high-performance model as a reference point.

In [ ]:
# CELL 11 — PLAINTEXT XGBOOST BASELINE

baseline_model = XGBBaseline(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.03,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.90,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=1.5,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=80
)

print("Starting plaintext XGBoost training...")

start_time = time.time()

baseline_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

baseline_training_time = time.time() - start_time

print(
    "\nPlaintext training time:",
    round(baseline_training_time, 2),
    "seconds"
)

In [ ]:
# CELL 12 — BASELINE EVALUATION

baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print(
    "\nPLAINTEXT XGBOOST ACCURACY:",
    round(baseline_accuracy * 100, 4),
    "%"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        baseline_predictions,
        target_names=[str(x) for x in target_encoder.classes_],
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, baseline_predictions))

## Part B — Concrete ML

The FHE model intentionally starts smaller than the plaintext baseline.
A smaller tree ensemble is much more practical for encrypted inference.

In [ ]:
# CELL 13 — CONCRETE ML XGBOOST

concrete_model = ConcreteXGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.90,
    objective="binary:logistic",
    random_state=42
)

print("Concrete ML XGBoost model created.")

In [ ]:
# CELL 14 — TRAIN CONCRETE ML MODEL

print("Training Concrete ML XGBoost...")

start_time = time.time()

concrete_model.fit(
    X_train,
    y_train
)

concrete_training_time = time.time() - start_time

print(
    "\nConcrete ML training time:",
    round(concrete_training_time, 2),
    "seconds"
)

In [ ]:
# CELL 15 — CONCRETE ML PLAINTEXT PREDICTION

concrete_plain_predictions = concrete_model.predict(X_test)

concrete_plain_predictions = (
    np.asarray(concrete_plain_predictions)
    .reshape(-1)
    .astype(int)
)

concrete_plain_accuracy = accuracy_score(
    y_test,
    concrete_plain_predictions
)

print(
    "Concrete ML plaintext accuracy:",
    round(concrete_plain_accuracy * 100, 4),
    "%"
)

## Part C — Compile the Concrete ML Model to FHE

The compilation step creates the FHE circuit and determines the quantization
needed for encrypted computation.

In [ ]:
# CELL 16 — COMPILE MODEL TO FHE

print("Compiling Concrete ML model to FHE...")

calibration_size = min(500, len(X_train))

calibration_data = X_train.iloc[
    :calibration_size
].to_numpy(dtype=np.float32)

compile_start = time.time()

quantized_module = concrete_model.compile(
    calibration_data
)

compile_time = time.time() - compile_start

print(
    "\nFHE compilation completed in:",
    round(compile_time, 2),
    "seconds"
)

In [ ]:
# CELL 17 — CHECK FHE CIRCUIT

print("FHE circuit:")
print(quantized_module.fhe_circuit)

In [ ]:
# CELL 18 — GENERATE FHE KEYS

print("Generating FHE keys...")

key_start = time.time()

quantized_module.keygen()

keygen_time = time.time() - key_start

print(
    "Key generation completed in:",
    round(keygen_time, 2),
    "seconds"
)

## Part D — Encrypted Inference

Start with only a few samples. FHE inference is substantially slower than
ordinary inference, so the complete test set should not be used initially.

In [ ]:
# CELL 19 — ENCRYPTED INFERENCE

FHE_TEST_SIZE = 10

X_fhe_test = X_test.iloc[
    :FHE_TEST_SIZE
].to_numpy(dtype=np.float32)

y_fhe_test = y_test[:FHE_TEST_SIZE]

print(
    "Running FHE inference on",
    FHE_TEST_SIZE,
    "samples..."
)

# Encryption
fhe_start = time.time()

encrypted_input = quantized_module.encrypt(
    X_fhe_test
)

encryption_time = time.time() - fhe_start

print(
    "\nEncryption time:",
    round(encryption_time, 4),
    "seconds"
)

# FHE inference
fhe_inference_start = time.time()

encrypted_prediction = quantized_module.run(
    encrypted_input
)

fhe_inference_time = time.time() - fhe_inference_start

print(
    "FHE inference time:",
    round(fhe_inference_time, 4),
    "seconds"
)

# Decryption
decrypt_start = time.time()

decrypted_prediction = quantized_module.decrypt(
    encrypted_prediction
)

decryption_time = time.time() - decrypt_start

print(
    "Decryption time:",
    round(decryption_time, 4),
    "seconds"
)

In [ ]:
# CELL 20 — VERIFY FHE PREDICTIONS

fhe_predictions = (
    np.asarray(decrypted_prediction)
    .reshape(-1)
    .astype(int)
)

print("FHE Predictions:")
print(fhe_predictions)

print("\nActual:")
print(y_fhe_test)

fhe_accuracy = accuracy_score(
    y_fhe_test,
    fhe_predictions
)

print(
    "\nFHE ACCURACY:",
    round(fhe_accuracy * 100, 2),
    "%"
)

In [ ]:
# CELL 21 — PLAINTEXT VS FHE COMPARISON

concrete_plain_test_predictions = (
    concrete_model.predict(X_fhe_test)
)

concrete_plain_test_predictions = (
    np.asarray(concrete_plain_test_predictions)
    .reshape(-1)
    .astype(int)
)

comparison = pd.DataFrame({
    "Actual": y_fhe_test,
    "Concrete_Plaintext": concrete_plain_test_predictions,
    "Concrete_FHE": fhe_predictions
})

display(comparison)

print(
    "\nPlaintext vs FHE predictions identical:",
    np.array_equal(
        concrete_plain_test_predictions,
        fhe_predictions
    )
)

In [ ]:
# CELL 22 — PERFORMANCE COMPARISON

print("\n")
print("=" * 60)
print("           FHE PERFORMANCE RESULTS")
print("=" * 60)

print(
    "\nPlaintext XGBoost accuracy :",
    round(baseline_accuracy * 100, 4),
    "%"
)

print(
    "Concrete ML accuracy       :",
    round(concrete_plain_accuracy * 100, 4),
    "%"
)

print(
    "FHE accuracy               :",
    round(fhe_accuracy * 100, 2),
    "%"
)

print(
    "\nFHE compilation time       :",
    round(compile_time, 2),
    "seconds"
)

print(
    "FHE key generation time    :",
    round(keygen_time, 2),
    "seconds"
)

print(
    "Encryption time            :",
    round(encryption_time, 4),
    "seconds"
)

print(
    "FHE inference time         :",
    round(fhe_inference_time, 4),
    "seconds"
)

print(
    "Decryption time            :",
    round(decryption_time, 4),
    "seconds"
)

print("=" * 60)

In [ ]:
# CELL 23 — FEATURE IMPORTANCE

feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": baseline_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        by="importance",
        ascending=False
    )
)

print("\nFeature Importance:\n")
print(feature_importance.to_string(index=False))

In [ ]:
# CELL 24 — SAVE MODELS

joblib.dump(
    baseline_model,
    "/content/cloud_xgboost_baseline.pkl"
)

joblib.dump(
    concrete_model,
    "/content/concrete_xgboost_model.pkl"
)

joblib.dump(
    target_encoder,
    "/content/target_encoder.pkl"
)

joblib.dump(
    encoders,
    "/content/feature_encoders.pkl"
)

joblib.dump(
    list(X.columns),
    "/content/feature_columns.pkl"
)

print("Models and preprocessing objects saved.")

In [ ]:
# CELL 25 — FHE PREDICTION FOR A NEW SAMPLE

sample = X_test.iloc[[0]].to_numpy(dtype=np.float32)
actual = y_test[0]

encrypted_sample = quantized_module.encrypt(sample)

encrypted_result = quantized_module.run(
    encrypted_sample
)

fhe_result = quantized_module.decrypt(
    encrypted_result
)

predicted = int(
    np.asarray(fhe_result).reshape(-1)[0]
)

actual_class = target_encoder.inverse_transform(
    [int(actual)]
)[0]

predicted_class = target_encoder.inverse_transform(
    [predicted]
)[0]

print("\n================================")
print("       FHE NEW SAMPLE")
print("================================")

print("Actual     :", actual_class)
print("Predicted  :", predicted_class)

print("================================")

In [ ]:
# CELL 26 — FINAL MODEL SUMMARY

print("\n")
print("=" * 70)
print("                 FINAL MODEL SUMMARY")
print("=" * 70)

print("\nDATASET")
print("-----------------------------")
print("Total samples       :", len(df))
print("Features            :", X.shape[1])
print("Classes             :", num_classes)

print("\nPLAINTEXT BASELINE")
print("-----------------------------")
print(
    "Accuracy            :",
    round(baseline_accuracy * 100, 4),
    "%"
)

print(
    "Training time       :",
    round(baseline_training_time, 2),
    "seconds"
)

print("\nCONCRETE ML")
print("-----------------------------")
print(
    "Plaintext accuracy  :",
    round(concrete_plain_accuracy * 100, 4),
    "%"
)

print(
    "Compilation time    :",
    round(compile_time, 2),
    "seconds"
)

print(
    "Key generation      :",
    round(keygen_time, 2),
    "seconds"
)

print("\nFHE")
print("-----------------------------")
print(
    "Test samples        :",
    FHE_TEST_SIZE
)

print(
    "FHE accuracy        :",
    round(fhe_accuracy * 100, 2),
    "%"
)

print(
    "Encryption time     :",
    round(encryption_time, 4),
    "seconds"
)

print(
    "Inference time      :",
    round(fhe_inference_time, 4),
    "seconds"
)

print(
    "Decryption time     :",
    round(decryption_time, 4),
    "seconds"
)

print("\n" + "=" * 70)